# Multi-Model Image Generation

Compare image generation across 3 AI models using the same prompt.

**Before running:** get a free Hugging Face token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and paste it below.

In [ ]:
!pip install -q Pillow requests matplotlib

In [ ]:
import requests, io, time
from PIL import Image
import matplotlib.pyplot as plt

HF_TOKEN = "paste-your-token-here"
HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"}

PROMPT = "a flat illustration of a smiling merchant reviewing analytics on a tablet in a retail store, minimal style, no text, no watermark"

models = {
    "Stable Diffusion XL": "stabilityai/stable-diffusion-xl-base-1.0",
    "Flux Schnell": "black-forest-labs/FLUX.1-schnell",
    "SD 3 Medium": "stabilityai/stable-diffusion-3-medium-diffusers",
}

class AuthError(Exception):
    pass

class RateLimitError(Exception):
    pass

def generate(model_id, prompt):
    url = f"https://router.huggingface.co/hf-inference/models/{model_id}"
    for attempt in range(3):
        try:
            r = requests.post(url, headers=HEADERS, json={"inputs": prompt}, timeout=120)
        except requests.RequestException as e:
            print(f"  ✗ Request failed: {e}")
            return None

        content_type = r.headers.get("content-type", "unknown")

        if r.status_code == 200:
            if "image" not in content_type:
                print(f"  ✗ Expected image, got {content_type}: {r.text[:300]}")
                return None
            try:
                return Image.open(io.BytesIO(r.content))
            except Exception as e:
                print(f"  ✗ Failed to decode image ({content_type}, {len(r.content)} bytes): {e}")
                return None
        if r.status_code == 401:
            raise AuthError("Invalid token. Check your HF_TOKEN above.")
        if r.status_code == 402:
            raise RateLimitError("Free API limit reached. Try again later.")
        if r.status_code == 503:
            try:
                wait = r.json().get("estimated_time", 30)
            except Exception:
                wait = 30
            print(f"  Model loading, waiting {wait:.0f}s (attempt {attempt + 1}/3)...")
            time.sleep(wait)
            continue

        print(f"  ✗ Error {r.status_code} ({content_type}):\n    {r.text[:500]}")
        return None
    return None

images = {}
try:
    for name, mid in models.items():
        print(f"Generating with {name}...")
        result = generate(mid, PROMPT)
        images[name] = result
        if result is not None:
            print("  ✓ Success")
except AuthError as e:
    print(f"\n⚠️  {e}\nGet a valid token at https://huggingface.co/settings/tokens")
except RateLimitError as e:
    print(f"\n⚠️  {e}")

valid = {k: v for k, v in images.items() if v is not None}

if not valid:
    if not images:
        pass
    else:
        print("\nNo images generated. The free API limit may have been reached — try again later.")
else:
    fig, axes = plt.subplots(1, len(valid), figsize=(8 * len(valid), 8))
    if len(valid) == 1: axes = [axes]
    for ax, (name, img) in zip(axes, valid.items()):
        ax.imshow(img)
        ax.set_title(name, fontsize=16, pad=20)
        ax.axis("off")
    fig.suptitle(f'Prompt: {PROMPT}', fontsize=18, color="#1a73e8", y=1.05)
    plt.tight_layout()
    plt.show()